# Lab A — CPU vs GPU Inference Baseline / CPU 与 GPU 推理性能基线
## Slide 33 · 同一个模型，两种 AMD 设备

**What will you learn? / 本节学什么？**
- Latency (P50/P95)、Throughput、Warm-up 的准确含义 What latency P50/P95, throughput and warm-up really mean
- 同一份官方 YOLO26l 在 AMD CPU 与 W7900 GPU 上的真实差异 The measured difference of the same official YOLO26l on the AMD CPU vs the W7900

**One key idea / 一个核心原理**
> 深度学习推理 = 海量矩阵乘法。GPU 的数千个并行核心天然适合这类工作负载，但“快多少”必须实测。Inference = huge matrix math. GPUs are built for it, but *how much faster* must be measured, not assumed.

In [ ]:
from pathlib import Path
import sys
REPO = Path.cwd()
if REPO.name == "notebooks":
    REPO = REPO.parent
sys.path.insert(0, str(REPO))
print("repo:", REPO)

## 1. Key metrics / 关键指标
- **Latency 延迟**: 处理 1 张图的耗时（P50 = 中位数，P95 = 95 分位，反映最差情况）
- **Throughput 吞吐**: 每秒处理多少张图（img/s）
- **Warm-up 预热**: 前几次运行会初始化内核/缓存，不计入统计
- 我们测量的是**端到端**时间：预处理 + 模型计算 + 后处理

Latency = time for one image (P50 median, P95 tail). Throughput = images/s. Warm-up runs initialize kernels and are excluded. We measure **end-to-end** time: preprocess + model + postprocess.

In [ ]:
import torch
from course.model import load_official_yolo26l
from course.benchmark import load_bench_images, pin_torch_threads, cgroup_cpu_quota

# Fair CPU config: PyTorch is fastest at 8 threads under this container's
# 16-core CPU quota (measured: 8->190ms, 64->516ms per image!)
print("torch CPU threads:", pin_torch_threads(8),
      "| cgroup CPU quota:", cgroup_cpu_quota(), "cores")
model = load_official_yolo26l()
images = load_bench_images()
IMG = images[:20]          # fixed image pool for this lab

## 2. Run the baseline: AMD CPU / 基线：AMD CPU
对同一张图片反复推理，取统计值。We repeatedly predict the same images and collect per-iteration latency samples.

In [ ]:
WARMUP, ITERS = 3, 10          # quick mode; run_all --profile full uses 10/50

from course.benchmark import bench, save_run, RunMeta

def predict_cpu(path):
    model.predict(source=str(path), imgsz=640, device="cpu",
                  verbose=False)

cpu_ms = bench(lambda: predict_cpu(IMG[0]), warmup=WARMUP, iters=ITERS, sync=False)
cpu_sum = save_run(RunMeta("labA_cpu_vs_gpu", "pytorch_cpu", "cpu",
                           "pytorch-rocm-cpu", warmup=WARMUP, iters=ITERS,
                           notes="notebook quick run"), cpu_ms)
print(f"CPU P50 {cpu_sum['stats']['p50_ms']:.1f} ms · "
      f"P95 {cpu_sum['stats']['p95_ms']:.1f} ms")

## 3. Change one thing: same model, same image → W7900 GPU / 只改一件事：换到 W7900
唯一变化是 `device="cuda"`（ROCm GPU）。GPU 计算是异步的，计时前后必须 `torch.cuda.synchronize()`——这正是共享工具 `bench()` 做的事。
The only change: `device="cuda"`. GPU calls are asynchronous, so the shared `bench()` helper synchronizes before/after each timed call.

In [ ]:
gpu_ms = bench(lambda: model.predict(source=str(IMG[0]), imgsz=640,
                                device="cuda", verbose=False),
               warmup=WARMUP, iters=ITERS, sync=True)
gpu_sum = save_run(RunMeta("labA_cpu_vs_gpu", "pytorch_rocm_gpu", "cuda(gfx1100)",
                           "pytorch-rocm", warmup=WARMUP, iters=ITERS,
                           notes="notebook quick run"), gpu_ms)
print(f"GPU P50 {gpu_sum['stats']['p50_ms']:.1f} ms · "
      f"P95 {gpu_sum['stats']['p95_ms']:.1f} ms")

## 4. Compare the results / 对比真实结果

In [ ]:
import pandas as pd

rows = []
for name, s in [("AMD CPU", cpu_sum), ("W7900 GPU", gpu_sum)]:
    st = s["stats"]
    rows.append({"device": name, "P50 (ms)": round(st["p50_ms"], 1),
                 "P95 (ms)": round(st["p95_ms"], 1),
                 "throughput (img/s)": round(1000 / st["p50_ms"], 1)})
df = pd.DataFrame(rows)
ratio = cpu_sum["stats"]["p50_ms"] / gpu_sum["stats"]["p50_ms"]
df["speedup vs CPU"] = ["1.00x", f"{ratio:.2f}x"]
display(df)

In [ ]:
from course.charts import chart_cpu_vs_gpu
chart_cpu_vs_gpu(cpu_sum, gpu_sum)

## 5. Why? / 为什么会这样？
YOLO26l 每张图要执行数千万次乘加。W7900 拥有 6144 个流处理器和 48GB 高带宽显存，可以同时执行成千上万路运算；CPU 虽然单核更强，但并行宽度小得多。
注意：我们比较的是**设备整体方案**（CPU 平台 vs GPU 平台），而不是纯粹软件优化——那是 Lab B 的主题。

YOLO26l performs tens of millions of MACs per image. The W7900 has 6144 stream processors and 48 GB of high-bandwidth memory; the CPU has far fewer but smarter cores. Note we compare **platforms**, not pure software runtimes — that is Lab B.

如果差距没有你想象的极端，想想：预处理/后处理仍在 CPU 上，小 batch 下 GPU 可能“吃不饱”。If the gap is smaller than expected, remember pre/post-processing still runs on the CPU and batch=1 may under-utilize the GPU.

## 6. Try it yourself / 学生练习
先写下你的预测，再把 `STUDENT_DEVICE` 改成 `"cpu"` 或 `"cuda"`、`STUDENT_ITERS` 改成 20，重新运行下面格子，观察 P50/P95 的变化。
Predict first, then set `STUDENT_DEVICE` and `STUDENT_ITERS` and re-run.

In [ ]:
STUDENT_DEVICE = "cuda"        # <-- change me: "cpu" or "cuda"
STUDENT_ITERS  = 10           # <-- change me
my_prediction  = "P50 will be ___ ms"   # <-- write your prediction

ms = bench(lambda: model.predict(source=str(IMG[1]), imgsz=640,
                     device=STUDENT_DEVICE, verbose=False),
           warmup=2, iters=STUDENT_ITERS, sync=(STUDENT_DEVICE == "cuda"))
from course.benchmark import stats_of
print("prediction :", my_prediction)
print("measured   :", {k: round(v, 1) for k, v in stats_of(ms).items()})

**Takeaway / 一句话总结**
> 硬件架构决定计算方式：同一份 YOLO26l，在 GPU 与 CPU 上的执行速度可以相差一个数量级——但具体多少，必须实测。
> Architecture decides how math is executed: the same YOLO26l can run an > order of magnitude differently across devices — measure it.